# Preprocessing and Training Data Development #



I'll be using the following variables for my machine learning prediction:
- Education Requirements
- Years of Experience
- Skills (top 15)

In [21]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns
import sklearn
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer


By the way, here's the top 15 skills:

In [12]:
# Read from top 15 skills text file and print to console
with open('../data/processed/top_15_skills.txt', 'r') as f:
    top_15_skills_text = f.read()
    print(top_15_skills_text)

# Convert the text to a list of skills
top_15_skills = top_15_skills_text.split('\n')
top_15_skills = [skill.split('. ')[1].split(': ')[0] for skill in top_15_skills[1:-1]]  # Extract skill names

Top 15 Skills Required for Popular Job Titles:
1. Pandas: 46.04%
2. LightGBM: 39.57%
3. Scikit-learn: 39.53%
4. Random Forest: 39.49%
5. Regression: 39.46%
6. XGBoost: 39.39%
7. NumPy: 39.35%
8. Supervised Learning: 39.29%
9. Feature Engineering: 39.24%
10. PyTorch: 22.15%
11. MLflow: 19.20%
12. AWS SageMaker: 19.17%
13. CI/CD: 19.09%
14. Triton Inference Server: 19.05%
15. Terraform: 19.00%



In [13]:
# reading in the data
job_postings = pd.read_csv('../data/processed/wrangled_american_cleaned_job_data.csv')

In [14]:
job_postings.head()

,Job Title,Company Name,Company Industry,Company Size,Job Location,Remote / Hybrid / On-site,Country,Salary Range,Experience Level,Employment Type,...,AI Specialization,Education Requirements,Years of Experience Required,Job Description,Posting Date,Benefits Offered,Company Rating,Number of Applicants,Job URL,Data Collection Timestamp
0,AI Agentic Engineer - Multiple USA locations,ClifyX,Technology,"1,001-5,000 employees","Bentonville, AR",Hybrid,United States,"$140,000 - $180,000",Senior,Full-time,...,Generative AI,"Master's in Computer Science, Machine Learning...",5,We are seeking a senior AI Engineer to lead ge...,2026-06-22,"Health insurance, 401(k), Unlimited PTO",4.2,42,https://www.linkedin.com/jobs/view/ai-agentic-...,2026-06-22T11:41:12.537989
1,AI Engineer,Casumo,Technology,"1,001-5,000 employees","Swieqi, Malta",Hybrid,United States,"$140,000 - $180,000",Senior,Full-time,...,Generative AI,"Master's in Computer Science, Machine Learning...",5,We are seeking a senior AI Engineer to lead ge...,2026-06-22,"Health insurance, 401(k), Unlimited PTO",4.2,42,https://mt.linkedin.com/jobs/view/ai-engineer-...,2026-06-22T11:41:12.540233
2,"Applied AI Engineer, Global Public Sector",Scale AI,Technology,"1,001-5,000 employees","Doha, Qatar",Hybrid,United States,"$140,000 - $180,000",Senior,Full-time,...,Generative AI,"Master's in Computer Science, Machine Learning...",5,We are seeking a senior AI Engineer to lead ge...,2026-06-22,"Health insurance, 401(k), Unlimited PTO",4.2,42,https://qa.linkedin.com/jobs/view/applied-ai-e...,2026-06-22T11:41:12.545900
3,"Machine Learning Engineer, Global Public Sector",Scale AI,Technology,"1,001-5,000 employees","Doha, Qatar",Hybrid,United States,"$140,000 - $180,000",Senior,Full-time,...,Generative AI,"Master's in Computer Science, Machine Learning...",5,We are seeking a senior AI Engineer to lead ge...,2026-06-22,"Health insurance, 401(k), Unlimited PTO",4.2,42,https://qa.linkedin.com/jobs/view/machine-lear...,2026-06-22T11:41:12.547144
4,AI Agent & Automation Developer,Cherry Bekaert,Technology,"1,001-5,000 employees","Augusta, GA",Hybrid,United States,"$140,000 - $180,000",Senior,Full-time,...,Generative AI,"Master's in Computer Science, Machine Learning...",5,We are seeking a senior AI Engineer to lead ge...,2026-06-22,"Health insurance, 401(k), Unlimited PTO",4.2,42,https://www.linkedin.com/jobs/view/ai-agent-au...,2026-06-22T11:41:12.548409


Let's start by making features out of Years of Experience and Education Requirements

First, let's do Education Requirements

In [15]:
job_postings[['Education Requirements', 'Years of Experience Required']]

,Education Requirements,Years of Experience Required
0,"Master's in Computer Science, Machine Learning...",5
1,"Master's in Computer Science, Machine Learning...",5
2,"Master's in Computer Science, Machine Learning...",5
3,"Master's in Computer Science, Machine Learning...",5
4,"Master's in Computer Science, Machine Learning...",5
...,...,...
8761,"PhD in Machine Learning, Computer Science, Sta...",6
8762,"Master's in Computer Science, Machine Learning...",3
8763,"Master's in Computer Science, Machine Learning...",5
8764,"Bachelor's in Data Science, Statistics, or Mat...",10


## Experience and education features

We'll keep required years as a number. 

In [16]:
job_postings['years_experience_required'] = pd.to_numeric(
    job_postings['Years of Experience Required'], errors='coerce'
)
if job_postings['years_experience_required'].isna().any():
    raise ValueError('Some experience requirements could not be converted to numbers')

## Education features

We'll turn the education requirement into three degree indicators for later modeling

In [17]:
degree_names = {"Bachelor's": 'bachelors', "Master's": 'masters', 'PhD': 'phd'}
job_postings['education_degree'] = (
    job_postings['Education Requirements']
    .str.extract(r"^(Bachelor's|Master's|PhD)", expand=False)
    .map(degree_names)
)
if job_postings['education_degree'].isna().any():
    raise ValueError('Some education requirements have an unrecognized degree')
education_columns = ['education_bachelors', 'education_masters', 'education_phd']
education_indicators = (
    pd.get_dummies(job_postings['education_degree'], prefix='education', dtype=int)
    .reindex(columns=education_columns, fill_value=0)
)
job_postings[education_columns] = education_indicators

In [22]:
# Use model_features with your salary target and skill features in later cells.
feature_columns = ['years_experience_required', *education_columns]
model_features = job_postings[feature_columns].copy()
model_features.head()

,years_experience_required,education_bachelors,education_masters,education_phd
0,5,0,1,0
1,5,0,1,0
2,5,0,1,0
3,5,0,1,0
4,5,0,1,0


In [24]:
model_education_features = model_features[education_columns].copy()
model_education_features.head()

,education_bachelors,education_masters,education_phd
0,0,1,0
1,0,1,0
2,0,1,0
3,0,1,0
4,0,1,0


In [26]:
model_years_experience_features = model_features[['years_experience_required']].copy()
model_years_experience_features.head()

,years_experience_required
0,5
1,5
2,5
3,5
4,5
